# Cloning data from the old database to have a clean slate

In [1]:
import duckdb

new_db_path = r"C:\Users\agand\Desktop\Personal_Coding\Finance_Projects\MultiFactorModel_LFQA\multi-factor-models\venv\crsp\new_factor_data.duckdb"
old_db_path = r"C:\Users\agand\Desktop\Personal_Coding\Finance_Projects\MultiFactorModel_LFQA\multi-factor-models\venv\crsp\crsp_dataset.duckdb"

conn = duckdb.connect(new_db_path)

# inline the path
conn.execute(f"ATTACH '{old_db_path}' AS old")

# sanity check
print(conn.execute("SHOW TABLES FROM old").fetchall())

tables_to_copy = ["crsp_clean"]
for t in tables_to_copy:
    conn.execute(f"CREATE TABLE {t} AS SELECT * FROM old.{t}")

conn.execute("DETACH old")
conn.close()




[('crsp_clean',), ('crsp_continuity',), ('crsp_gaps',), ('crsp_table',), ('factor_exposures_monthly',), ('factor_exposures_z',), ('factor_returns_monthly',), ('factors_daily',), ('factors_monthly',), ('monthly_base',), ('monthly_with_mom',), ('returns_monthly',), ('v_crsp_base',)]


# Factor Work

## Connection to DB

In [ ]:
import duckdb

new_db_path = r"C:\Users\agand\Desktop\Personal_Coding\Finance_Projects\MultiFactorModel_LFQA\multi-factor-models\venv\crsp\new_factor_data.duckdb"
conn = duckdb.connect(new_db_path)
conn.execute("SHOW TABLES;").fetch_df()

,name
0,crsp_clean
1,daily_factors_table
2,factor_top30_monthly_returns
3,monthly_base
4,monthly_factors_table
5,monthly_with_mom
6,returns_monthly


In [23]:
conn.execute("""
SELECT * FROM crsp_clean  
ORDER BY date ASC, cusip       
LIMIT 10
""").fetch_df()

,cusip,date,permno,permco,cfacpr,cfacshr,shrout,prc,ret,retx,vol,ask,askhi,bid,bidlo
0,00036020,2002-01-02,76868,10817,11.390625,11.390625,8666.0,22.00,-0.100940,-0.100940,157053.0,22.00,24.20,21.85,21.90
1,00036110,2002-01-02,54594,20000,1.000000,1.000000,26958.0,9.20,0.021088,0.021088,58000.0,9.20,9.23,9.01,8.99
2,00037520,2002-01-02,88953,41444,1.038169,1.000000,7388.0,9.89,0.051010,0.051010,27400.0,NaN,10.00,NaN,9.72
3,00078810,2002-01-02,86828,16482,1.000000,1.000000,12887.0,2.15,-0.088983,-0.088983,56865.0,2.15,2.35,2.13,2.11
4,00086T10,2002-01-02,85452,15750,2.000000,2.000000,7466.0,30.48,0.033220,0.033220,30754.0,30.57,30.55,30.36,28.61
5,00087B10,2002-01-02,76276,5817,1.000000,1.000000,10681.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,00088630,2002-01-02,50906,2902,0.142857,0.142857,792000.0,4.68,0.017391,0.017391,7754418.0,4.68,4.77,4.67,4.54
7,00088E10,2002-01-02,76290,10587,1.000000,1.000000,55882.0,1.85,-0.010695,-0.010695,38014.0,1.88,1.95,1.84,1.78
8,00088N10,2002-01-02,86817,16481,1.000000,1.000000,38185.0,0.31,0.148148,0.148148,37835.0,0.32,0.31,0.29,0.29
9,00088U10,2002-01-02,82544,14114,1.000000,1.000000,7296.0,2.95,-0.016667,-0.016667,2873.0,2.97,3.07,2.95,2.81


## Generate Factors

In [16]:
conn.execute("""
DROP TABLE IF EXISTS daily_factors_table;
CREATE TABLE daily_factors_table AS
SELECT
c.cusip,
c.date,
c.permno,
c.permco,
c.prc,
c.shrout,
c.vol,
c.ask,
c.bid,
c.askhi,
c.bidlo,
c.ret,
c.retx,
CASE WHEN c.prc IS NULL OR c.shrout IS NULL
    THEN NULL
    ELSE c.prc * c.shrout * 1000
END AS market_cap,
    
CASE WHEN c.vol IS NULL OR c.shrout IS NULL OR c.shrout = 0
    THEN NULL
    ELSE c.vol / c.shrout
    END AS turnover,

CASE 
    WHEN c.ask IS NULL OR c.bid IS NULL OR (c.ask + c.bid ) = 0
    THEN NULL
    ELSE (c.ask - c.bid) / ((c.ask + c.bid) / 2.0)
END AS bid_ask_spread

FROM crsp_clean c
ORDER BY c.cusip, c.date;
""")

In [27]:
conn.execute("""
SELECT * FROM daily_factors_table
ORDER BY date ASC, cusip
LIMIT 5
""").fetch_df()

,cusip,date,permno,permco,prc,shrout,vol,ask,bid,askhi,bidlo,ret,retx,market_cap,turnover,bid_ask_spread
0,00036020,2002-01-02,76868,10817,22.00,8666.0,157053.0,22.00,21.85,24.20,21.90,-0.100940,-0.100940,190652000.0,18.122894,0.006842
1,00036110,2002-01-02,54594,20000,9.20,26958.0,58000.0,9.20,9.01,9.23,8.99,0.021088,0.021088,248013600.0,2.151495,0.020868
2,00037520,2002-01-02,88953,41444,9.89,7388.0,27400.0,NaN,NaN,10.00,9.72,0.051010,0.051010,73067320.0,3.708717,NaN
3,00078810,2002-01-02,86828,16482,2.15,12887.0,56865.0,2.15,2.13,2.35,2.11,-0.088983,-0.088983,27707050.0,4.412586,0.009346
4,00086T10,2002-01-02,85452,15750,30.48,7466.0,30754.0,30.57,30.36,30.55,28.61,0.033220,0.033220,227563680.0,4.119207,0.006893


## Monthly Aggregation for three first factors
- market cap
- turnover
- BA spread

In [26]:
conn.execute("""
            DROP TABLE IF EXISTS monthly_factors_table;
            CREATE TABLE monthly_factors_table AS
            WITH tagged AS (
            SELECT
                cusip,
                date,
                date_trunc('month', date) AS month,
                market_cap,
                turnover,
                bid_ask_spread
            FROM daily_factors_table
            )
             
            SELECT
                cusip,
            month AS month_start,
            (DATE_TRUNC('month', month) + INTERVAL 1 MONTH - INTERVAL 1 DAY)::DATE AS month_end,
            ARG_MAX(market_cap, date) AS market_cap_month_end,
            AVG(turnover) AS turnover_month_avg,
            AVG(bid_ask_spread) AS spread_month_avg,
            COUNT (*) AS trading_days_in_month
            FROM tagged
            GROUP BY cusip, month
            ORDER BY cusip, month;
             """)

In [30]:
conn.execute("""
SELECT * FROM monthly_factors_table
ORDER BY month_start ASC, cusip
LIMIT 5
""").fetch_df()

,cusip,month_start,month_end,market_cap_month_end,turnover_month_avg,spread_month_avg,trading_days_in_month
0,00036020,2002-01-01,2002-01-31,170715867.0,2.944402,0.009464,21
1,00036110,2002-01-01,2002-01-31,230987400.0,3.175799,0.016575,21
2,00037520,2002-01-01,2002-01-31,69044300.0,3.486583,0.009512,21
3,00078810,2002-01-01,2002-01-31,18557280.0,3.278856,0.022227,21
4,00086T10,2002-01-01,2002-01-31,204419080.0,4.796404,0.008468,21


## Getting rolling factors (returns and momentum)

In [36]:
conn.execute("""
DROP TABLE IF EXISTS returns_monthly;
CREATE TABLE returns_monthly AS
WITH m AS (
    SELECT
    cusip,
    DATE_TRUNC('month', date)::DATE AS month_start,
    EXP(SUM(LOG(1 + ret)) FILTER (WHERE ret IS NOT NULL AND ret > -0.999999)) - 1 AS ret_m
    FROM crsp_clean
    GROUP BY 1,2
    
    )
SELECT * FROM m
ORDER BY cusip, month_start;
""")

In [ ]:
conn.execute("""
SELECT * FROM returns_monthly
ORDER BY month_start ASC
LIMIT 10
""").fetch_df()

,cusip,month_start,ret_m
0,00036020,2002-01-01,-0.089881
1,00036110,2002-01-01,-0.018735
2,00037520,2002-01-01,-0.035913
3,00078810,2002-01-01,-0.193096
4,00086T10,2002-01-01,-0.031869
5,00087B10,2002-01-01,NaN
6,00088630,2002-01-01,0.025995
7,00088E10,2002-01-01,-0.104590
8,00088N10,2002-01-01,-0.067266
9,00088U10,2002-01-01,0.049090


In [37]:
conn.execute("""
DROP TABLE IF EXISTS monthly_base;
CREATE TABLE monthly_base AS
    SELECT
    fm.cusip,
    fm.month_start,
    fm.month_end,
    fm.market_cap_month_end,
    fm.turnover_month_avg,
    fm.spread_month_avg,
    rm.ret_m
FROM monthly_factors_table AS fm
    LEFT JOIN returns_monthly AS rm
    USING(cusip, month_start)
ORDER BY cusip, month_start;
""")

In [42]:
conn.execute("""
SELECT * FROM monthly_base
ORDER BY month_start ASC, cusip
LIMIT 5
""").fetch_df()

,cusip,month_start,month_end,market_cap_month_end,turnover_month_avg,spread_month_avg,ret_m
0,00036020,2002-01-01,2002-01-31,170715867.0,2.944402,0.009464,-0.089881
1,00036110,2002-01-01,2002-01-31,230987400.0,3.175799,0.016575,-0.018735
2,00037520,2002-01-01,2002-01-31,69044300.0,3.486583,0.009512,-0.035913
3,00078810,2002-01-01,2002-01-31,18557280.0,3.278856,0.022227,-0.193096
4,00086T10,2002-01-01,2002-01-31,204419080.0,4.796404,0.008468,-0.031869


In [ ]:
conn.execute("""
DROP TABLE IF EXISTS monthly_with_mom;
CREATE TABLE monthly_with_mom AS
WITH w AS (
  SELECT
    cusip,
    month_start,
    market_cap_month_end,
    turnover_month_avg,
    spread_month_avg,
    ret_m,
    
    COUNT(ret_m) OVER (
      PARTITION BY cusip
      ORDER BY month_start
      ROWS BETWEEN 12 PRECEDING AND 2 PRECEDING
    ) AS win_count,
    SUM(LOG(1 + ret_m)) OVER (
      PARTITION BY cusip
      ORDER BY month_start
      ROWS BETWEEN 12 PRECEDING AND 2 PRECEDING
    ) AS win_logsum
  FROM monthly_base
)
SELECT
  cusip,
  month_start,
  market_cap_month_end,
  turnover_month_avg,
  spread_month_avg,
  ret_m,
  
  CASE WHEN win_count = 11 THEN EXP(win_logsum) - 1 ELSE NULL END AS mom_12_1
FROM w
ORDER BY cusip, month_start;
""")


In [47]:
conn.execute("""
SELECT * FROM monthly_with_mom
ORDER BY month_start DESC, cusip
LIMIT 5
""").fetch_df()

,cusip,month_start,market_cap_month_end,turnover_month_avg,spread_month_avg,ret_m,mom_12_1
0,00032Q10,2023-12-01,4.954252e+07,32.611961,0.005145,-0.340079,-0.189922
1,00036020,2023-12-01,6.001568e+09,5.357582,0.000830,0.074541,0.006502
2,00036110,2023-12-01,2.215387e+09,9.479900,0.000535,-0.044527,0.046875
3,00038020,2023-12-01,5.149022e+09,30.400653,0.000417,0.000362,0.069637
4,00039J10,2023-12-01,6.261608e+08,10.257881,0.000388,0.003477,0.008889


In [ ]:
conn.close()